# Repeated Switching in Household Healthy Purchasing Behavior

**Six-category specification, including milk.** The notebook produces three small-multiple panels with gray lines, blue/red points, subtle regime shading and a shared empirical threshold. No life-event variables are used.

**Measure.** Annual healthier food-spending share is the sum of spending on fresh fruits and vegetables, low-sugar cereal, healthy snacks, healthier beverages, fat-free yogurt and skimmed milk, divided by annual total food spending. Shares are calculated from all eligible shopping trips.

**Descriptive switch.** A switch is a crossing of the *pooled median of eligible annual household healthy-spending shares*. To avoid counting tiny movements near the threshold, the endpoints of a switch must fall on opposite sides of a configurable exclusion band (default ±2.5 percentage points). Years within the band are ambiguous and do not independently establish a regime; crossings through the band are allowed only across consecutive eligible years. Each selected household must show **at least three switches**, at least eight consecutive observed years, and at least twelve shopping trips in each year. This is an illustrative selection, not evidence of latent state identification, strict periodicity, or population prevalence.

**Privacy.** All data, individual-household figure outputs, identifier mappings and screening files are written to the private local data directory, not to GitHub. Figures can be moved into the public repository only after confirming that sharing is permitted.

In [ ]:
from pathlib import Path
import json
import os
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# Keep research data and household-level images outside the Git repository.
DATA_ROOT = Path(r"C:\Users\rwen13\Documents\healthy_purchase_data")
SOURCE = DATA_ROOT / "processed" / "hmm_input" / "hmm_estimation_panel.parquet"
OUTPUT_DIR = DATA_ROOT / "model_outputs" / "model_free_evidence" / "switching_trajectories_six_category"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

YEAR_START, YEAR_END = 2008, 2017
N_EXAMPLES = 3
MIN_CONSECUTIVE_YEARS = 8
MIN_ANNUAL_TRIPS = 12
MIN_SWITCHES = 3
EXCLUSION_BAND_PP = 2.5  # Both endpoint regimes must be this far from the pooled median.
RANDOM_SEED = 20261031
DUCKDB_THREADS = min(4, max(1, (os.cpu_count() or 2)//2))
CACHE = OUTPUT_DIR / "annual_six_category_spending.parquet"
CACHE_META = OUTPUT_DIR / "annual_six_category_spending.meta.json"

HEALTHY_SPENDING_COLUMNS = [
    "fresh_spending",           # Fruits and Vegetables
    "low_sugar_cereal_spending",
    "healthy_snack_spending",
    "healthy_beverage_spending",
    "skimmed_yogurt_spending",
    "skimmed_milk_spending",    # Milk is included.
]
assert MIN_SWITCHES >= 3 and EXCLUSION_BAND_PP > 0


## 1. Construct household-year shares from trip-level purchase data

In [ ]:
if not SOURCE.is_file():
    raise FileNotFoundError(f"Six-category HMM trip panel not found: {SOURCE}")

# Metadata locks the cache to the current source version and aggregation rules.
meta = {
    "source": str(SOURCE.resolve()),
    "size": SOURCE.stat().st_size,
    "mtime_ns": SOURCE.stat().st_mtime_ns,
    "columns": HEALTHY_SPENDING_COLUMNS,
    "years": [YEAR_START, YEAR_END],
    "version": "six-category-annual-share-v1",
}
if CACHE.exists() and CACHE_META.exists() and json.loads(CACHE_META.read_text(encoding="utf-8")) == meta:
    annual = pd.read_parquet(CACHE)
    print(f"Reused annual cache: {len(annual):,} household-years")
else:
    con = duckdb.connect()
    try:
        con.execute(f"SET threads={DUCKDB_THREADS}")
        con.execute("SET memory_limit='2GB'")
        con.execute(f"SET temp_directory='{(OUTPUT_DIR / 'duckdb_temp').as_posix()}'")
        source_sql = SOURCE.as_posix().replace("'", "''")
        con.execute(f"CREATE VIEW trips AS SELECT * FROM read_parquet('{source_sql}')")
        available = {row[0] for row in con.execute("DESCRIBE trips").fetchall()}
        required = {"hhid", "panel_year", "trip_id", "purchase_date", "food_spend", *HEALTHY_SPENDING_COLUMNS}
        missing = required - available
        if missing:
            raise ValueError(f"Required columns missing from source: {sorted(missing)}")
        # Scan the large trip file once; do not include event or region variables.
        spending_expression = " + ".join(f"COALESCE(\"{c}\", 0)" for c in HEALTHY_SPENDING_COLUMNS)
        annual = con.execute(f"""
            SELECT CAST(hhid AS VARCHAR) AS hhid,
                   CAST(panel_year AS INTEGER) AS panel_year,
                   COUNT(*) AS n_trips,
                   SUM(food_spend) AS food_spend,
                   SUM({spending_expression}) AS healthy_spend
              FROM trips
             WHERE panel_year BETWEEN ? AND ?
             GROUP BY 1, 2
        """, [YEAR_START, YEAR_END]).fetchdf()
    finally:
        con.close()
    if annual.duplicated(["hhid", "panel_year"]).any():
        raise ValueError("Annual aggregation produced duplicate household-years")
    annual["Y"] = annual.healthy_spend / annual.food_spend.where(annual.food_spend > 0)
    invalid = annual.Y.notna() & ~annual.Y.between(-1e-10, 1+1e-10)
    if invalid.any():
        raise ValueError(f"Healthy spending share outside [0,1] in {int(invalid.sum())} household-years")
    annual.to_parquet(CACHE, index=False)
    CACHE_META.write_text(json.dumps(meta, indent=2), encoding="utf-8")
    print(f"Created annual cache: {len(annual):,} household-years")

print(f"Households: {annual.hhid.nunique():,}; Years: {annual.panel_year.min()}–{annual.panel_year.max()}")


## 2. Find consecutive annual observations and count genuine regime crossings

In [ ]:
required = ["hhid", "panel_year", "n_trips", "Y"]
valid = annual.loc[
    annual.Y.notna() & annual.Y.between(0, 1)
    & (annual.n_trips >= MIN_ANNUAL_TRIPS), required
].sort_values(["hhid", "panel_year"]).copy()

# A single pooled threshold is shared across panels; it is not an HMM-estimated state boundary.
threshold = float(valid.Y.median())
band = EXCLUSION_BAND_PP / 100.0
if not (0 < threshold-band < threshold+band < 1):
    raise ValueError("Exclusion band exceeds the available healthy-share range")

# Invalid or missing household-years interrupt spells, rather than being bridged.
new_spell = valid.hhid.ne(valid.hhid.shift()) | valid.panel_year.diff().ne(1)
valid["spell_id"] = new_spell.cumsum().astype("int64")

# Robust regime definition: 1 = clearly above; -1 = clearly below; 0 = ambiguous.
valid["regime"] = np.select(
    [valid.Y >= threshold + band, valid.Y <= threshold - band],
    [1, -1], default=0
).astype("int8")

def count_switches(g):
    """Count crossings only after reaching the opposite clear regime.

    Ambiguous years do not create switches, and cannot bridge a missing calendar year
    because this function is applied separately to each consecutive-year spell.
    """
    clear = g.regime.to_numpy()
    clear = clear[clear != 0]
    return int(np.sum(clear[1:] != clear[:-1])) if len(clear) > 1 else 0

spell_stats = valid.groupby("spell_id", sort=False).agg(
    hhid=("hhid", "first"), n_years=("panel_year", "size"),
    first_year=("panel_year", "min"), last_year=("panel_year", "max"),
    annual_range_pp=("Y", lambda x: float((x.max()-x.min())*100)),
)
spell_stats["switches"] = valid.groupby("spell_id", sort=False).apply(
    count_switches, include_groups=False
).astype(int)

# Each household contributes only its longest consecutive valid spell.
longest = (spell_stats.sort_values(["n_years", "first_year"], ascending=[False, True])
           .drop_duplicates("hhid"))
eligible = longest.loc[
    (longest.n_years >= MIN_CONSECUTIVE_YEARS)
    & (longest.switches >= MIN_SWITCHES)
].copy()
summary = pd.DataFrame([
    {"stage": "All households", "households": annual.hhid.nunique()},
    {"stage": "At least one valid household-year", "households": valid.hhid.nunique()},
    {"stage": "Longest consecutive spell >= minimum years", "households": int((longest.n_years >= MIN_CONSECUTIVE_YEARS).sum())},
    {"stage": f"Also >= {MIN_SWITCHES} robust crossings", "households": len(eligible)},
])
display(summary)
summary.to_csv(OUTPUT_DIR / "screening_counts.csv", index=False)
print(f"Pooled median threshold: {threshold*100:.2f}%; crossing band: ±{EXCLUSION_BAND_PP:g} pp")
if len(eligible) < N_EXAMPLES:
    raise ValueError(f"Only {len(eligible)} households meet the preset rules; {N_EXAMPLES} needed. No threshold was silently relaxed.")

# Prefer the most clear-cut 3+ crossing examples; use a deterministic random tie break.
# This is purposive illustration, NOT a representative household sample.
eligible = eligible.sort_values("hhid").copy()
eligible["tie_break"] = np.random.default_rng(RANDOM_SEED).random(len(eligible))
selected = eligible.sort_values(
    ["switches", "annual_range_pp", "n_years", "tie_break"],
    ascending=[False, False, False, True]
).head(N_EXAMPLES).copy()
selected["example_label"] = [f"Household {chr(65+i)}" for i in range(N_EXAMPLES)]
examples = valid.loc[valid.spell_id.isin(selected.index)].copy()
examples["example_label"] = examples.spell_id.map(selected.example_label)
selected.to_csv(OUTPUT_DIR / "selected_households_private.csv", index_label="spell_id")
examples.to_parquet(OUTPUT_DIR / "selected_trajectories_private.parquet", index=False)
display(selected[["example_label", "n_years", "first_year", "last_year", "switches", "annual_range_pp"]])


## 3. Produce the six-panel figure

In [ ]:
plt.rcParams.update({
    "font.family": "serif", "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42,
})
blue, red, gray = "#2266A5", "#CF3636", "#777777"
fig, axes = plt.subplots(1, 3, figsize=(15, 5.6), sharex=True, sharey=True)

# Shared scale: preserve zero as a reference and the same height in each panel.
max_display = min(100, max(50, int(np.ceil(examples.Y.max()*100/10))*10))
for ax, (_, item) in zip(axes.flat, selected.iterrows()):
    s = examples.loc[examples.example_label.eq(item.example_label)].sort_values("panel_year")
    x, y = s.panel_year.to_numpy(), 100*s.Y.to_numpy()
    ax.axhspan(0, threshold*100, color="#FBEAEA", alpha=0.85, zorder=0)
    ax.axhspan(threshold*100, max_display, color="#E9F3FF", alpha=0.85, zorder=0)
    ax.axhline(threshold*100, linestyle="--", color="#666666", linewidth=1, zorder=1)
    ax.plot(x, y, color=gray, linewidth=1.8, zorder=2)
    # All observations keep their true values. Ambiguous points appear gray.
    colors = np.where(s.regime.to_numpy()==1, blue,
                      np.where(s.regime.to_numpy()==-1, red, gray))
    ax.scatter(x, y, c=colors, s=45, edgecolor="white", linewidth=0.5, zorder=3)
    ax.set_title(f"{item.example_label}  (Switches = {int(item.switches)})", fontsize=12, fontweight="bold")
    ax.set_ylim(0, max_display)
    ax.set_xlim(YEAR_START-0.3, YEAR_END+0.3)
    ax.set_xticks(np.arange(YEAR_START, YEAR_END+1, 2))
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))
    ax.grid(axis="y", color="#DDDDDD", linewidth=0.6)
    ax.set_axisbelow(True)

fig.suptitle("Repeated Switching in Household Healthy Purchasing Behavior", fontsize=18, fontweight="bold", y=0.97)
fig.text(0.5, 0.89, "Annual healthy share of food spending (six food categories)", ha="center", fontsize=11)
fig.supxlabel("Calendar Year", fontsize=11, y=0.19)
fig.supylabel("Healthy Share of Food Spending (%)", fontsize=11, x=0.02)
legend = [
    Line2D([0],[0], color="none", marker="o", markerfacecolor=blue, markeredgecolor="white", markersize=8, label="Higher healthy purchasing"),
    Line2D([0],[0], color="none", marker="o", markerfacecolor=red, markeredgecolor="white", markersize=8, label="Lower healthy purchasing"),
    Line2D([0],[0], color="none", marker="o", markerfacecolor=gray, markeredgecolor="white", markersize=8, label="Near threshold"),
    Line2D([0],[0], color="#666666", linestyle="--", label=f"Pooled median = {threshold*100:.1f}%"),
]
fig.legend(handles=legend, loc="lower center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 0.11), fontsize=9)
fig.text(0.055, 0.025,
         f"Notes: Selected households have ≥{MIN_CONSECUTIVE_YEARS} consecutive years, ≥{MIN_ANNUAL_TRIPS} trips/year and ≥{MIN_SWITCHES} crossings "
         f"between regimes outside a ±{EXCLUSION_BAND_PP:g} pp band. Six categories include Fruits and Vegetables, "
         "cereal, snacks, beverages, yogurt, and milk. Descriptive regimes are not HMM latent states.",
         ha="left", va="bottom", fontsize=8, color="#444444", wrap=True)
fig.subplots_adjust(left=0.075, right=0.985, top=0.75, bottom=0.30, wspace=0.16)
for extension in ("png", "pdf"):
    fig.savefig(OUTPUT_DIR / f"repeated_switching_three_households.{extension}", dpi=300, bbox_inches="tight")
plt.show()

settings = {
    "category_label": "Fruits and Vegetables",
    "healthy_spending_columns": HEALTHY_SPENDING_COLUMNS,
    "denominator": "sum of annual trip food_spend",
    "threshold_definition": "Pooled median among valid household-years",
    "median_threshold": threshold,
    "exclusion_band_pp": EXCLUSION_BAND_PP,
    "min_switches": MIN_SWITCHES,
    "min_consecutive_years": MIN_CONSECUTIVE_YEARS,
    "min_annual_trips": MIN_ANNUAL_TRIPS,
    "eligible_households": len(eligible),
    "selection": "Purposive: prioritize switch count, then range, spell length, seeded tie-break",
}
(OUTPUT_DIR / "figure_settings.json").write_text(json.dumps(settings, indent=2), encoding="utf-8")
print(f"Private figure and audit outputs: {OUTPUT_DIR}")
